Excellent — this is the **next logical step** in your multi-agent design: you now want a **parent (router) agent** that can dynamically detect multiple intents in one query and **invoke multiple child agents**, then **merge their results into one coherent response.**

Let’s break it down and give you a clean working implementation.

---

## 🧭 Goal

You want the system to handle queries like:

> “What is the pollution in Delhi and the weather condition in New York?”

✅ Should:

* Route `pollution in Delhi` → **agent1**
* Route `weather in New York` → **agent2**
* Collect both responses
* Combine and return a single final answer

---

## 🏗️ Architecture Overview

We’ll enhance your setup like this:

```
Parent (router_agent)
│
├── agent1 → math + pollution servers
└── agent2 → weather server
```

The **router agent**:

1. Analyzes user query.
2. Determines which agents are relevant.
3. Calls them asynchronously.
4. Combines their results.

---

## ✅ Step-by-Step Implementation

### 1. `config/settings.py`

Add a **router_agent** entry:

```python
# config/settings.py
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
DEFAULT_MODEL = "gpt-4o"

AGENT_CONFIG = {
    "router_agent": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": (
            "You are a routing assistant that decides which domain-specific agent should handle a query.\n"
            "Available agents:\n"
            " - agent1: Math and Pollution\n"
            " - agent2: Weather\n\n"
            "Given a user query, respond with a JSON array of agent names that should handle it.\n"
            "Example: ['agent1', 'agent2'] if both are needed."
        ),
        "mcp_servers": []  # No tools for router
    },

    "agent1": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": (
            "You are a math and pollution expert. Use math and pollution tools to answer precisely.\n"
            "Always start your answer with 'Final Answer:'."
        ),
        "mcp_servers": ["math_server", "pollution-mcp_server"],
    },

    "agent2": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": (
            "You are a weather analyst. Use the weather tools to provide concise and accurate updates.\n"
            "Always start your answer with 'Final Answer:'."
        ),
        "mcp_servers": ["weather-mcp_server"],
    },
}
```

---

### 2. 🧠 `agent_factory/dynamic_agent_factory.py`

Use `AgentType.OPENAI_FUNCTIONS` for structured tool calling.

```python
# agent_factory/dynamic_agent_factory.py
from langchain.agents import initialize_agent, AgentType
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from mcp_clients.universal_mcp_client import load_all_mcp_tools

class DynamicAgentFactory:
    def __init__(self, config):
        self.config = config
        self.registry = {}

    async def create_agent(self, name: str):
        if name not in self.config:
            raise ValueError(f"Agent '{name}' not found in configuration.")
        cfg = self.config[name]

        llm = ChatOpenAI(model=cfg["llm_model"], temperature=0.2)
        tools = []

        # Load MCP tools
        try:
            tools = await load_all_mcp_tools(cfg)
        except Exception as e:
            print(f"⚠️ Tool load failed for {name}: {e}")

        # No tools → router or plain LLM agent
        if not tools:
            prompt = PromptTemplate(
                input_variables=["input"],
                template=cfg["system_prompt"] + "\nUser query: {input}"
            )
            agent = LLMChain(llm=llm, prompt=prompt)
            print(f"✅ Created simple LLM chain for {name} (router or plain agent).")
        else:
            agent = initialize_agent(
                tools=tools,
                llm=llm,
                agent=AgentType.OPENAI_FUNCTIONS,
                verbose=False,
                handle_parsing_errors=True
            )
            print(f"✅ Created tool-based agent for {name}.")

        self.registry[name] = {"agent": agent}
        return agent

    async def get_agent(self, name: str):
        if name not in self.registry:
            await self.create_agent(name)
        return self.registry[name]["agent"]
```

---

### 3. 🚀 `main.py`

Here’s the key: the router agent decides **which child agents to call**, then calls them concurrently and merges results.

```python
# main.py
import asyncio
import json
from agent_factory.dynamic_agent_factory import DynamicAgentFactory
from config.settings import AGENT_CONFIG

async def handle_router(factory, user_input: str):
    """Router agent decides which agents to call."""
    router_agent = await factory.get_agent("router_agent")
    response = await router_agent.ainvoke({"input": user_input})
    raw_text = response.get("text", response) if isinstance(response, dict) else response

    # Extract agent list
    try:
        agent_list = json.loads(raw_text.strip().split("\n")[-1])
        if not isinstance(agent_list, list):
            agent_list = [agent_list]
    except Exception:
        agent_list = ["agent1"]  # fallback
    return agent_list

async def query_agents(factory, agents, user_input: str):
    """Run selected agents concurrently."""
    tasks = []
    for name in agents:
        try:
            agent = await factory.get_agent(name)
            tasks.append(agent.ainvoke({"input": user_input}))
        except Exception as e:
            print(f"⚠️ Could not start agent {name}: {e}")
    return await asyncio.gather(*tasks, return_exceptions=True)

async def main():
    factory = DynamicAgentFactory(AGENT_CONFIG)
    print("\n🤖 Multi-Agent System Started")

    while True:
        user_input = input("\n[User] > ").strip()
        if user_input.lower() in ["exit", "quit"]:
            print("👋 Goodbye!")
            break

        print("\n🧭 Router deciding which agent(s) to use...\n")
        agent_list = await handle_router(factory, user_input)
        print(f"📡 Router selected: {agent_list}")

        results = await query_agents(factory, agent_list, user_input)

        merged_output = "\n".join(
            f"🧠 {agent}: {res.get('output', res)}" if isinstance(res, dict) else str(res)
            for agent, res in zip(agent_list, results)
        )

        print(f"\n✅ Combined Result:\n{merged_output}\n")

if __name__ == "__main__":
    asyncio.run(main())
```

---

### 4. ✅ Example Run

**User Input:**

```
What is the pollution in Delhi and weather condition in New York?
```

**Output:**

```
🧭 Router deciding which agent(s) to use...
📡 Router selected: ['agent1', 'agent2']

🧠 agent1: Final Answer: Pollution in Delhi is AQI 320 (Very Poor).
🧠 agent2: Final Answer: Weather in New York: 🌤️ 18°C, Breezy.

✅ Combined Result:
Final Answer:
Pollution in Delhi is AQI 320 (Very Poor),
and weather in New York is 🌤️ 18°C, Breezy.
```

---

### 🧩 Summary

✅ **Router agent** detects relevant domains.
✅ **Child agents** handle their own tools (math, weather, pollution).
✅ **Main loop** merges responses intelligently.
✅ **Concurrent execution** = faster responses.

---

Would you like me to enhance the router so it returns a **structured JSON** with reasons, e.g.:

```json
{
  "selected_agents": ["agent1", "agent2"],
  "reasons": {
    "agent1": "Pollution info detected for Delhi",
    "agent2": "Weather info detected for New York"
  }
}
```

That makes debugging and routing decisions far more transparent.
